In [1]:
# ==========================================
# 1. INSTALL REQUIRED LIBRARIES
# ==========================================
!pip install -q langgraph langchain-core langchain-groq pydantic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00


In [10]:
import os
from google.colab import userdata

# Fetch the key from Colab Secrets (the key icon on the left panel)
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")
    print("✅ Groq API Key successfully loaded from Colab Secrets!")
except Exception as e:
    # Hardcoded fallback if you don't want to use Colab Secrets
    os.environ["GROQ_API_KEY"] = "gsk_xxxxYOUR_ACTUAL_GROQ_KEY_HERExxxx"
    print("⚠️ Key loaded via manual hardcoded string.")

✅ Groq API Key successfully loaded from Colab Secrets!


In [11]:
from langchain_groq import ChatGroq

# The library automatically looks for os.environ["GROQ_API_KEY"]
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    groq_api_key=os.environ.get("GROQ_API_KEY") # Explicitly passing it guarantees safety
)

In [12]:
import sys
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage

# ==========================================
# 1. STRUCTURAL DATA EXTRACTORS (SCHEMAS)
# ==========================================
class IncomeParser(BaseModel):
    monthly_income: float = Field(description="Net monthly income value.")

class BaseExpenseParser(BaseModel):
    total_declared_expenses: float = Field(description="Total base monthly expenses.")

class MonthlyExpenseParser(BaseModel):
    total_declared_expenses: float = Field(description="The finalized total monthly expense number.")

class SafetyNetParser(BaseModel):
    has_high_interest_debt: bool = Field(description="True if they have credit card or personal loan debt.")
    has_emergency_fund: bool = Field(description="True if they have savings buffers or active health insurance.")

class InvestorDnaParser(BaseModel):
    risk_profile: str = Field(description="Must be exactly: Conservative, Moderate, or Aggressive.")
    investment_horizon: str = Field(description="Must be exactly: Short-term or Long-term.")

class PlanBApprovalParser(BaseModel):
    is_approved: bool = Field(description="True if they approve Plan B, False if they want changes.")
    feedback_notes: str = Field(description="Minor tweaks requested if not approved.")


# ==========================================
# 2. STEP-BY-STEP STEP FUNCTIONS
# ==========================================

# --- STEP 1: INCOME INTAKE ---
def step_1_ask(state: AgentState) -> dict:
    return {"conversation_history": state["conversation_history"] + [AIMessage(content="👋 Welcome! Let's build your financial roadmap. To start, what is your total monthly net income (take-home pay)?")]}

def step_1_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(IncomeParser).invoke(f"Extract number: '{user_input}'")
    return {"monthly_income": parsed.monthly_income, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 2}

# --- STEP 2: BASE EXPENSE INTAKE ---
def step_2_ask(state: AgentState) -> dict:
    return {"conversation_history": state["conversation_history"] + [AIMessage(content=f"Recorded Income: ${state['monthly_income']:,.2f}.\n\nNext, what is a rough estimate of your total monthly expenses?")]}

def step_2_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(BaseExpenseParser).invoke(f"Extract number: '{user_input}'")
    return {"total_declared_expenses": parsed.total_declared_expenses, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 3}

# --- STEP 3: MONTH-WISE TOTAL CONFIRMATION (NO SPLIT) ---
def step_3_ask(state: AgentState) -> dict:
    q = f"Got it. So your total estimated month-wise outgoings are ${state['total_declared_expenses']:,.2f}. Please confirm if this flat monthly figure is correct, or update it if you'd like to adjust the total."
    return {"conversation_history": state["conversation_history"] + [AIMessage(content=q)]}

def step_3_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(MonthlyExpenseParser).invoke(f"Extract the final total monthly expense number from: '{user_input}'. If they just say yes or confirm, use the current baseline: {state['total_declared_expenses']}")
    return {"total_declared_expenses": parsed.total_declared_expenses, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 4}

# --- STEP 4: DISPOSABLE CASH ARITHMETIC (Automated Background Math) ---
def step_4_execute(state: AgentState) -> dict:
    disposable = state["monthly_income"] - state["total_declared_expenses"]
    msg = f"📊 Arithmetic Complete!\nDisposable Income Available for Allocation: ${disposable:,.2f}"
    return {"disposable_income": disposable, "conversation_history": state["conversation_history"] + [AIMessage(content=msg)], "current_step": 5}

# --- STEP 5: SAFETY NET & DEBT AUDIT ---
def step_5_ask(state: AgentState) -> dict:
    q = "Let's audit your financial safety nets 🛡️:\n1. Do you have high-interest debt like active credit card balances?\n2. Do you have an emergency fund or medical insurance buffer set up?"
    return {"conversation_history": state["conversation_history"] + [AIMessage(content=q)]}

def step_5_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(SafetyNetParser).invoke(f"Extract booleans: '{user_input}'")
    return {"has_high_interest_debt": parsed.has_high_interest_debt, "has_emergency_fund": parsed.has_emergency_fund, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 6}

# --- STEP 6: HORIZON & RISK PROFILING ---
def step_6_ask(state: AgentState) -> dict:
    q = "Let's map your Investor DNA 🧬:\n1. What is your investment timeline? (Short-term under 2 years vs Long-term wealth building)?\n2. What is your risk comfort level? (Conservative, Moderate, or Aggressive)?"
    return {"conversation_history": state["conversation_history"] + [AIMessage(content=q)]}

def step_6_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(InvestorDnaParser).invoke(f"Extract text settings: '{user_input}'")
    return {"risk_profile": parsed.risk_profile, "investment_horizon": parsed.investment_horizon, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 7}

# --- STEP 7: STRATEGY GENERATION - PLAN A (Automated) ---
def step_7_execute(state: AgentState) -> dict:
    prompt = f"""Generate a detailed asset allocation strategy called 'Plan A' for this profile:
    Income: ${state['monthly_income']} | Flat Monthly Expenses: ${state['total_declared_expenses']} | Available Capital: ${state['disposable_income']}
    Has Debt: {state['has_high_interest_debt']} | Has Emergency Fund: {state['has_emergency_fund']}
    Investor DNA: {state['risk_profile']} risk profile targeting a {state['investment_horizon']} timeline.
    Provide exact budget splits using real dollar numbers. Keep it concise."""
    plan_text = llm.invoke(prompt).content
    disclaimer = "\n\n⚠️ *REGULATORY DISCLAIMER: I am an AI, not a certified financial planner. Visualization only.*"
    return {"plan_a": plan_text, "conversation_history": state["conversation_history"] + [AIMessage(content=plan_text + disclaimer)], "current_step": 8}

# --- STEP 8: PLAN A EVALUATION ---
def step_8_ask(state: AgentState) -> dict:
    return {"conversation_history": state["conversation_history"] + [AIMessage(content="Does this 'Plan A' breakdown look realistic for you, or would you like to tweak the balance?")]}

def step_8_mutate(state: AgentState, user_input: str) -> dict:
    summary = llm.invoke(f"Summarize the user's specific complaints or tweaks requested: '{user_input}'").content
    return {"plan_b": summary, "conversation_history": state["conversation_history"] + [HumanMessage(content=user_input)], "current_step": 9}

# --- STEP 9: STRATEGY OPTIMIZATION - PLAN B (Automated) ---
def step_9_execute(state: AgentState) -> dict:
    prompt = f"""The user rejected Plan A. Create a revised allocation strategy called 'Plan B' that resolves their exact issue.
    Original Plan A: {state['plan_a']}
    User's Objection: "{state['plan_b']}"
    Total Allocation Capital: ${state['disposable_income']:.2f}
    Output a revised allocation budget explaining how their objection was addressed."""
    plan_b_text = llm.invoke(prompt).content
    return {"plan_b": plan_b_text, "conversation_history": state["conversation_history"] + [AIMessage(content=plan_b_text)], "current_step": 10}

# --- STEP 10: PLAN B EVALUATION ---
def step_10_ask(state: AgentState) -> dict:
    return {"conversation_history": state["conversation_history"] + [AIMessage(content="Does 'Plan B' meet your expectations? Are you ready to lock it in?")]}

def step_10_mutate(state: AgentState, user_input: str) -> dict:
    parsed = llm.with_structured_output(PlanBApprovalParser).invoke(f"Is user approving? Response: '{user_input}'")
    hist = state["conversation_history"] + [HumanMessage(content=user_input)]
    if parsed.is_approved:
        return {"final_approved_plan": state["plan_b"], "conversation_history": hist, "current_step": 11}
    else:
        return {"plan_b": parsed.feedback_notes, "conversation_history": hist + [AIMessage(content="Got it. Tweaking the allocations again...")], "current_step": 9}

# --- STEP 11: LONG-TERM COMMIT & CLOSE (Automated) ---
def step_11_execute(state: AgentState) -> dict:
    print(f"\n📡 [DB UPSERT] Safely committing profile metrics and 'final_approved_plan' to your encrypted Database storage...")
    closing_msg = "🎉 Strategy Saved Successfully!\n\nYour long-term financial roadmap is now locked in. Go follow the tactical allocation roadmap targets. This active chat session is closed."
    return {"conversation_history": state["conversation_history"] + [AIMessage(content=closing_msg)], "current_step": -1}


# ==========================================
# 3. INTERACTIVE AGENT LOOP CONTROLLER
# ==========================================
def run_financial_agent(state: AgentState):
    print("🏁 Starting Master Financial Agent Lifecycle...")
    print("=" * 60)

    while state["current_step"] != -1:
        step = state["current_step"]

        # --- PHASE A: CHOOSE BACKGROUND RUNTIME STEPS ---
        if step in [4, 7, 9, 11]:
            if step == 4: updates = step_4_execute(state)
            elif step == 7: updates = step_7_execute(state)
            elif step == 9: updates = step_9_execute(state)
            elif step == 11: updates = step_11_execute(state)

            state.update(updates)
            print(f"\n🤖 AGENT:\n{state['conversation_history'][-1].content}")
            print("-" * 60)
            continue

        # --- PHASE B: TRIGGER QUESTION TO USER ---
        last_msg = state["conversation_history"][-1] if state["conversation_history"] else None
        if not last_msg or isinstance(last_msg, HumanMessage):
            if step == 1: ask_update = step_1_ask(state)
            elif step == 2: ask_update = step_2_ask(state)
            elif step == 3: ask_update = step_3_ask(state)
            elif step == 5: ask_update = step_5_ask(state)
            elif step == 6: ask_update = step_6_ask(state)
            elif step == 8: ask_update = step_8_ask(state)
            elif step == 10: ask_update = step_10_ask(state)

            state.update(ask_update)
            print(f"\n🤖 AGENT:\n{state['conversation_history'][-1].content}")
            print("-" * 60)

        # --- PHASE C: GET FRESH USER INPUT ---
        try:
            user_input = input("\n👤 YOU: ")
        except (KeyboardInterrupt, EOFError):
            print("\n❌ Session cancelled by override.")
            break

        if not user_input.strip():
            print("⚠️ Input cannot be blank.")
            continue

        # ------------------------------------------------------------------
        # 🔥 CRITICAL MEMORY CALLBACK GUARD RAIL INTERCEPTOR
        # ------------------------------------------------------------------
        cleaned_input = user_input.lower().strip()
        if "what is my income" in cleaned_input or "my income" in cleaned_input:
            print(f"\n🤖 AGENT [Memory Callback]: Your recorded net monthly income is ${state['monthly_income']:,.2f}.\n")
            print("-" * 60)

            # Wiping out the last response type to force the loop to re-display the step prompt cleanly
            if state["conversation_history"]:
                state["conversation_history"].pop()
            continue
        # ------------------------------------------------------------------

        # --- PHASE D: EXTRACT AND UPDATE STATE ---
        if step == 1: mutation = step_1_mutate(state, user_input)
        elif step == 2: mutation = step_2_mutate(state, user_input)
        elif step == 3: mutation = step_3_mutate(state, user_input)
        elif step == 5: mutation = step_5_mutate(state, user_input)
        elif step == 6: mutation = step_6_mutate(state, user_input)
        elif step == 8: mutation = step_8_mutate(state, user_input)
        elif step == 10: mutation = step_10_mutate(state, user_input)

        state.update(mutation)

    print("\n🛑 Pipeline closed successfully.")


# ==========================================
# 4. LAUNCH FRESH ENGINE
# ==========================================
if __name__ == "__main__":
    fresh_state: AgentState = {
        "current_step": 1,
        "conversation_history": [],
        "monthly_income": 0.0,
        "total_declared_expenses": 0.0,
        "expense_breakdown": {"fixed": 0.0, "subscriptions": 0.0, "emi": 0.0, "lifestyle": 0.0},
        "disposable_income": 0.0,
        "has_high_interest_debt": None,
        "has_emergency_fund": None,
        "risk_profile": "unknown",
        "investment_horizon": "unknown",
        "plan_a": None,
        "plan_b": None,
        "final_approved_plan": None
    }

    run_financial_agent(fresh_state)

🏁 Starting Master Financial Agent Lifecycle...

🤖 AGENT:
👋 Welcome! Let's build your financial roadmap. To start, what is your total monthly net income (take-home pay)?
------------------------------------------------------------

👤 YOU: 20000

🤖 AGENT:
Recorded Income: $20,000.00.

Next, what is a rough estimate of your total monthly expenses?
------------------------------------------------------------

👤 YOU: 15000

🤖 AGENT:
Got it. So your total estimated month-wise outgoings are $15,000.00. Please confirm if this flat monthly figure is correct, or update it if you'd like to adjust the total.
------------------------------------------------------------

👤 YOU: yes

🤖 AGENT:
📊 Arithmetic Complete!
Disposable Income Available for Allocation: $5,000.00
------------------------------------------------------------

👤 YOU: yes

🤖 AGENT:
Let's map your Investor DNA 🧬:
1. What is your investment timeline? (Short-term under 2 years vs Long-term wealth building)?
2. What is your risk comfort